In [ ]:
# ========================================

import pandas as pd
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import re

class SeedSegmentator:
    def __init__(self):
        # Parametri per la segmentazione basata su colore (HSV)
        # Range per il ROSSO
        self.lower_red1 = np.array([0, 50, 50])    # Rosso basso
        self.upper_red1 = np.array([10, 255, 255])
        self.lower_red2 = np.array([170, 50, 50])  # Rosso alto
        self.upper_red2 = np.array([180, 255, 255])

        # Range per il VERDE CHIARO (radici) - molto più permissivo
        self.lower_green = np.array([25, 5, 10])    # Verde molto pallido/grigiastro
        self.upper_green = np.array([100, 255, 255]) # Range molto ampio

        # Parametri per il filtraggio dei contorni
        self.min_area = 800      # Area minima AUMENTATA dopo il collegamento
        self.max_area = 50000    # Area massima
        self.min_circularity = 0.05  # per forme allungate

        # Dimensioni target per la CNN
        self.target_size = (256, 256)  # Modifica secondo le tue necessità
        
    def create_seed_mask(self, image, timestamp):  # ADDED: timestamp for dynamic dilatation
        """Crea una maschera per identificare sia le aree rosse che verdi chiare"""
        # Converti in HSV per una migliore segmentazione del colore
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Crea maschere per il ROSSO (semi)
        red_mask1 = cv2.inRange(hsv, self.lower_red1, self.upper_red1)
        red_mask2 = cv2.inRange(hsv, self.lower_red2, self.upper_red2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)

        # Crea maschera per il VERDE CHIARO (radici) - molto permissiva
        green_mask = cv2.inRange(hsv, self.lower_green, self.upper_green)

        # STRATEGIA: Prima dilatare le maschere per collegarle, poi combinarle per via di come esce la radice

        # per timestamp avanti aumento
        dilate_size = 15 if timestamp <= 50 else 25
        kernel_red_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_size, dilate_size))
        red_dilated = cv2.dilate(red_mask, kernel_red_dilate, iterations=2)

        kernel_green_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        green_dilated = cv2.dilate(green_mask, kernel_green_dilate, iterations=1)

        # Combina le maschere dilatate
        combined_mask = cv2.bitwise_or(red_dilated, green_dilated)

        # Operazioni morfologiche per connettere tutto
        kernel_connect_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))  # Larger kernel
        combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, kernel_connect_large, iterations=3)
        # Riempie tutti i buchi interni
        kernel_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
        combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, kernel_fill)

        # Operazione finale di pulizia leggera consigliata 
        kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_OPEN, kernel_clean)

        return combined_mask, red_mask, green_mask

    def filter_contours(self, contours):
        """Filtra i contorni per includere semi+radici"""
        valid_contours = []

        for contour in contours:
            area = cv2.contourArea(contour)
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = max(w, h) / min(w, h) if min(w, h) > 0 else 0

            perimeter = cv2.arcLength(contour, True)
            circularity = 4 * np.pi * area / (perimeter * perimeter) if perimeter > 0 else 0

            # Filtro principale per area
            if area < self.min_area or area > self.max_area:
                continue

            # Filtro per forme ragionevoli
            if aspect_ratio > 10.0:
                continue

            # Filtro di circolarità molto permissivo
            if circularity >= self.min_circularity:
                valid_contours.append(contour)
                continue

            # Test aggiuntivo: se ha area sufficiente, accetta comunque
            if area > self.min_area * 1.5:
                valid_contours.append(contour)

        return valid_contours

    def custom_sort_seeds(self, contours):
        """
        Ordina i semi secondo una specifica disposizione spaziale per 10 semi.
        1. Il più in alto (seme 1)
        2. Sinistra nella seconda riga (seme 2)
        3. Destra nella seconda riga (seme 3)
        4, 5, 6, 7. La riga successiva, da sinistra a destra
        8. Sinistra nell'ultima area
        9. Centro in basso
        10. Destra sopra il 9
        """
        if len(contours) != 10:
            return self.sort_contours_left_to_right_top_to_bottom(contours)

        # Calcola i centroidi di tutti i contorni
        centroids = []
        for contour in contours:
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
            else:
                x, y, w, h = cv2.boundingRect(contour)
                cx = x + w//2
                cy = y + h//2
            centroids.append({'cx': cx, 'cy': cy, 'contour': contour})

        sorted_seeds = []
        available_centroids = centroids.copy()

        # 1. Trova il seme più in alto (seme 1)
        seed_1 = min(available_centroids, key=lambda c: c['cy'])
        sorted_seeds.append(seed_1['contour'])
        available_centroids.remove(seed_1)

        # 2. Trova i semi 2 e 3 (seconda riga: sinistra e destra)
        by_y = sorted(available_centroids, key=lambda c: c['cy'])
        second_row_candidates = by_y[:2]
        second_row_sorted = sorted(second_row_candidates, key=lambda c: c['cx'])

        seed_2 = second_row_sorted[0]  # Il più a sinistra
        seed_3 = second_row_sorted[1]  # Il più a destra

        sorted_seeds.append(seed_2['contour'])
        sorted_seeds.append(seed_3['contour'])
        available_centroids.remove(seed_2)
        available_centroids.remove(seed_3)

        # 3. Trova i semi 4, 5, 6, 7 (terza riga: da sinistra a destra)
        by_y_remaining = sorted(available_centroids, key=lambda c: c['cy'])
        third_row_candidates = by_y_remaining[:4]
        third_row_sorted = sorted(third_row_candidates, key=lambda c: c['cx'])

        for seed in third_row_sorted:
            sorted_seeds.append(seed['contour'])
            available_centroids.remove(seed)

        # 4. Ordina gli ultimi 3 semi (8, 9, 10)
        remaining_three = available_centroids

        # Seme 8: quello più a sinistra
        seed_8 = min(remaining_three, key=lambda c: c['cx'])
        sorted_seeds.append(seed_8['contour'])
        remaining_three.remove(seed_8)

        # Seme 9: quello più in basso dei due rimanenti
        seed_9 = max(remaining_three, key=lambda c: c['cy'])
        sorted_seeds.append(seed_9['contour'])
        remaining_three.remove(seed_9)

        # Seme 10: l'ultimo rimasto
        seed_10 = remaining_three[0]
        sorted_seeds.append(seed_10['contour'])

        return sorted_seeds

    def sort_contours_left_to_right_top_to_bottom(self, contours):
        """Ordina i contorni da sinistra a destra, dall'alto verso il basso (fallback)"""
        if not contours:
            return contours

        # Calcola il centroide di ogni contorno
        centroids = []
        for contour in contours:
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                centroids.append((cx, cy, contour))
            else:
                # Fallback: usa il bounding rect center
                x, y, w, h = cv2.boundingRect(contour)
                centroids.append((x + w//2, y + h//2, contour))

        # Ordina prima per Y (dall'alto al basso), poi per X (da sinistra a destra)
        # Usa una tolleranza per le righe (semi sulla stessa riga potrebbero avere Y leggermente diversi)
        def sort_key(item):
            cx, cy, _ = item
            # Raggruppa per righe usando tolleranza di 50 pixel
            row = cy // 50
            return (row, cx)

        sorted_centroids = sorted(centroids, key=sort_key)
        return [contour for _, _, contour in sorted_centroids]

    def extract_filename_info(self, filename):
        """
        Estrae informazioni dal nome del file per generare nomi consistenti.
        Esempi di pattern supportati:
        - "piastrina_scan1_99_p1.png" -> scan_part="scan1", mezzora="99", piastrina_num=1
        - "scan1_piastra2.jpg" -> scan_part="scan1", mezzora="2", piastrina_num=2
        - "piastra_3.png" -> scan_part="scan1", mezzora="3", piastrina_num=3
        - "image_p4.jpg" -> scan_part="scan1", mezzora="4", piastrina_num=4
        """
        # Rimuovi l'estensione
        base_name = Path(filename).stem.lower()  # lower for case insensitivity

        # Pattern per estrarre scan, mezzora e piastrina
        patterns = [
            r'(?:piastrina_)?scan(\d+)_(\d+)_p(\d+)',  # piastrina_scan1_99_p1 or scan1_99_p1
            r'scan(\d+).*piastra(\d+)',  # scan1_piastra2
            r'scan(\d+).*p(\d+)',        # scan1_p2
            r'piastra[_\s]*(\d+)',       # piastra_2 o piastra2
            r'p(\d+)',                   # p2
            r'(\d+)'                     # solo numero
        ]

        scan_part = "scan1"  # default
        mezzora = None
        piastrina_num = 1    # default

        for pattern in patterns:
            match = re.search(pattern, base_name)
            if match:
                groups = match.groups()
                if len(groups) == 3:  # New pattern with mezzora
                    scan_part = f"scan{groups[0]}"
                    mezzora = groups[1]  # string
                    piastrina_num = int(groups[2])
                elif len(groups) == 2:  # scan e piastrina
                    scan_part = f"scan{groups[0]}"
                    piastrina_num = int(groups[1])
                elif len(groups) == 1:  # solo piastrina
                    piastrina_num = int(groups[0])
                break

        # If mezzora not set, fallback to piastrina_num (for compatibility)
        if mezzora is None:
            mezzora = str(piastrina_num)

        return scan_part, mezzora, piastrina_num

    def extract_seed_roi(self, image, contour, padding=10):
        """Estrae la ROI di un singolo seme con padding"""
        # Ottieni il bounding rectangle
        x, y, w, h = cv2.boundingRect(contour)

        # Aggiungi padding
        x = max(0, x - padding)
        y = max(0, y - padding)
        w = min(image.shape[1] - x, w + 2 * padding)
        h = min(image.shape[0] - y, h + 2 * padding)

        # Estrai la ROI
        roi = image[y:y+h, x:x+w]

        # Crea maschera per questo contour specifico
        mask = np.zeros((h, w), dtype=np.uint8)
        contour_shifted = contour - [x, y]  # Sposta il contour alle coordinate della ROI
        cv2.fillPoly(mask, [contour_shifted], 255)

        # Applica la maschera per isolare solo il seme
        roi_masked = cv2.bitwise_and(roi, roi, mask=mask)

        # Converti in RGBA con trasparenza
        b, g, r = cv2.split(roi_masked)
        rgba = cv2.merge([b, g, r, mask])

        return rgba, mask

    def preprocess_for_cnn(self, seed_image):
        """Preprocessa il seme per l'input della CNN"""
        # Se è RGBA, rimuovi il canale alpha per il resize
        if seed_image.shape[2] == 4:
            bgr = seed_image[:, :, :3]
            alpha = seed_image[:, :, 3]
        else:
            bgr = seed_image
            alpha = None

        # Resize mantenendo l'aspect ratio
        h, w = bgr.shape[:2]
        if h > w:
            new_h = self.target_size[1]
            new_w = int(w * new_h / h)
        else:
            new_w = self.target_size[0]
            new_h = int(h * new_w / w)

        resized = cv2.resize(bgr, (new_w, new_h))

        # Crea immagine quadrata con padding nero
        square_img = np.zeros((self.target_size[1], self.target_size[0], 3), dtype=np.uint8)

        # Calcola offset per centrare
        y_offset = (self.target_size[1] - new_h) // 2
        x_offset = (self.target_size[0] - new_w) // 2

        square_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

        # Normalizza per la CNN (0-1)
        normalized = square_img.astype(np.float32) / 255.0

        return normalized

    def segment_seeds(self, image_path, output_dir, visualize=False): 
        """Funzione principale per segmentare i semi"""
        # Carica l'immagine
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Impossibile caricare l'immagine: {image_path}")

        # Crea directory di output
        Path(output_dir).mkdir(parents=True, exist_ok=True)

        # Estrai informazioni dal nome del file
        filename = Path(image_path).name
        scan_part, mezzora, piastrina_num = self.extract_filename_info(filename)

        
        timestamp = int(mezzora) if mezzora.isdigit() else 0  
        combined_mask, red_mask, green_mask = self.create_seed_mask(image, timestamp)

        # Trova contorni
        contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Filtra contorni validi
        valid_contours = self.filter_contours(contours)

        # Usa ordinamento personalizzato se abbiamo 10 semi, altrimenti standard
        if len(valid_contours) == 10:
            valid_contours = self.custom_sort_seeds(valid_contours)
        else:
            valid_contours = self.sort_contours_left_to_right_top_to_bottom(valid_contours)

        # Estrai ogni seme con il nuovo naming convention
        seeds_for_cnn = []
        seed_areas = []  

        for i, contour in enumerate(valid_contours):
            # Estrai ROI del seme
            pixel_area = cv2.contourArea(contour)
            seed_areas.append(pixel_area)  # FIXED: Append each area
            seed_rgba, mask = self.extract_seed_roi(image, contour)

            # Commentato per non salvare i singoli semi trasparenti
            # out_name = f"{scan_part}_{mezzora}_p{piastrina_num}_seed{i+1:02d}.png"
            # seed_path = os.path.join(output_dir, out_name)
            # cv2.imwrite(seed_path, seed_rgba)

            # Preprocessa per CNN
            cnn_input = self.preprocess_for_cnn(seed_rgba)
            seeds_for_cnn.append(cnn_input)

            # Salva la versione preprocessata (quella che finisce con _cnn.png) PER OGNI TIMEPOINT
            cnn_name = f"{scan_part}_{mezzora}_p{piastrina_num}_seed{i+1:02d}_cnn.png"
            cnn_path = os.path.join(output_dir, cnn_name)
            cv2.imwrite(cnn_path, (cnn_input * 255).astype(np.uint8))  # Sempre salvato ora

        # Visualizzazione risultati (disabilitata di default)
        if visualize and len(valid_contours) > 0:
            base_name = f"{scan_part}_{mezzora}_p{piastrina_num}"
            # Commentato per non salvare le visualizzazioni
            self.visualize_results(image, combined_mask, red_mask, green_mask, valid_contours, output_dir, base_name)

        return seeds_for_cnn, valid_contours, seed_areas

    def visualize_results(self, original, combined_mask, red_mask, green_mask, contours, output_dir, base_name):
        """Visualizza solo la maschera combinata e l'immagine con contorni verdi numerati"""
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Maschera combinata (bianco su nero)
        combined_mask_bin = (combined_mask > 0).astype('uint8')
        axes[0].imshow(combined_mask_bin, cmap='gray')
        axes[0].set_title('Maschera Combinata (bianco su nero)')
        axes[0].axis('off')

        # Immagine originale con contorni verdi e numerazione
        result_img = original.copy()
        cv2.drawContours(result_img, contours, -1, (0, 255, 0), 2)

        for i, contour in enumerate(contours):
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                cv2.putText(result_img, str(i+1), (cx-10, cy+5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        axes[1].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Semi+Radici Identificati ({len(contours)})')
        axes[1].axis('off')

        plt.tight_layout()
        viz_path = os.path.join(output_dir, f"{base_name}_visualization.png")
        plt.savefig(viz_path, dpi=150, bbox_inches='tight')
        plt.close()

    def debug_color_detection(self, image_path, output_dir="debug_colors"):
        """Funzione di debug per visualizzare cosa viene catturato dai range di colore"""
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Impossibile caricare l'immagine: {image_path}")

        Path(output_dir).mkdir(parents=True, exist_ok=True)

        # Converti in HSV
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Crea maschere separate
        red_mask1 = cv2.inRange(hsv, self.lower_red1, self.upper_red1)
        red_mask2 = cv2.inRange(hsv, self.lower_red2, self.upper_red2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)
        green_mask = cv2.inRange(hsv, self.lower_green, self.upper_green)
        combined_mask = cv2.bitwise_or(red_mask, green_mask)

        # Salva le maschere per debug
        base_name = Path(image_path).stem
        cv2.imwrite(f"{output_dir}/{base_name}_red_mask.png", red_mask)
        cv2.imwrite(f"{output_dir}/{base_name}_green_mask.png", green_mask)
        cv2.imwrite(f"{output_dir}/{base_name}_combined_raw.png", combined_mask)

        return red_mask, green_mask, combined_mask

from pathlib import Path
import os
import re

def process_all_piastrine(base_path=r"C:\Users\silve\Documents\Projects\Semi\piastrine", output_dir="segmented_seeds_ordered", target_piastrina=1):
    segmentator = SeedSegmentator()
    all_seeds_for_cnn = []
    processed_files = []
    all_seed_data = []
    base_path = Path(base_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']
    piastrina_files = sorted([f for f in base_path.iterdir() if f.suffix.lower() in image_extensions])

    print(f" Trovate {len(piastrina_files)} piastrine da processare.\n")

    for idx, file_path in enumerate(piastrina_files, 1):
        print(f" [{idx}/{len(piastrina_files)}] Inizio elaborazione: {file_path.name}")
        try:
            # Estrai informazioni per decidere se visualizzare
            scan_part, mezzora, piastrina_num = segmentator.extract_filename_info(file_path.name)
            timestamp = int(mezzora) if mezzora.isdigit() else 0

            # UPDATED: Visualize ONLY if timestamp is multiple of 25 AND piastrina_num == target_piastrina
            visualize = (timestamp % 25 == 0) and (piastrina_num == target_piastrina)
            if visualize:
                print(f"DEBUG: Saving visualization for {file_path.name} (timestamp={timestamp}, piastrina={piastrina_num})")

            seeds, contours, areas = segmentator.segment_seeds(str(file_path), str(output_dir), visualize=visualize)

            # Aggiungi info semi per CNN
            for i, (seed, area) in enumerate(zip(seeds, areas)):
                seed_info = {
                    'data': seed,
                    'file_name': file_path.name,
                    'scan': scan_part,
                    'piastra': piastrina_num,
                    'n_seme': i + 1,
                    'global_id': f"{scan_part}_{mezzora}_p{piastrina_num}_seed{i+1:02d}"
                }
                all_seeds_for_cnn.append(seed_info)
                seed_data = seed_info.copy()
                seed_data['timestamp'] = mezzora  
                seed_data['pixel_area'] = area    
                del seed_data['data']  
            processed_files.append({
                'file_name': file_path.name,
                'scan': scan_part,
                'piastra': piastrina_num,
                'seeds_count': len(seeds),
                'path': str(file_path)
            })

            print(f"✅ Fine {file_path.name}: {len(seeds)} semi trovati. Aree calcolate: {areas}\n")

        except Exception as e:
            print(f" Errore con {file_path.name}: {str(e)}\n")
            continue

    print(f" Completato: {len(processed_files)} piastrine processate, {len(all_seeds_for_cnn)} semi totali estratti.\n")
       
    df = pd.DataFrame(all_seed_data)[['scan', 'piastra', 'n_seme', 'timestamp', 'pixel_area', 'global_id']]
    df = df.sort_values(['scan', 'piastra', 'n_seme', 'timestamp'])
    csv_path = os.path.join(output_dir, 'seed_areas.csv')
    df.to_csv(csv_path, index=False)
    print(f"💾 DataFrame salvato in: {csv_path}")

   
    growth_df = compute_growth_percentage(df)
    growth_df.to_csv(os.path.join(output_dir, 'seed_growth.csv'), index=False)
    print("DataFrame con aree e crescita percentuale:\n", growth_df.head())
    return all_seeds_for_cnn, processed_files, df


def compute_growth_percentage(df, baseline_timestamp='0'):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    
    # Group by scan, piastra, n_seme
    growth_df = df.copy()
    
    growth_df = growth_df.sort_values(['scan', 'piastra', 'n_seme', 'timestamp'])
    growth_df['percent_growth'] = growth_df.groupby(['scan', 'piastra', 'n_seme'])['pixel_area'].transform(
        lambda x: (x - x.iloc[0]) / x.iloc[0] * 100 if len(x) > 1 else 0
    )
    
    def enforce_non_decreasing(group):
        prev_area = None
        for idx, row in group.iterrows():
            if prev_area is not None and row['pixel_area'] < prev_area:
                print(f"WARNING: Area decreased for {row['global_id']} at t={row['timestamp']}. Setting to previous: {prev_area}")
                group.at[idx, 'pixel_area'] = prev_area
            prev_area = group.at[idx, 'pixel_area']
        return group
    
    growth_df = growth_df.groupby(['scan', 'piastra', 'n_seme']).apply(enforce_non_decreasing).reset_index(drop=True)
    
    growth_df['percent_growth'] = growth_df.groupby(['scan', 'piastra', 'n_seme'])['pixel_area'].transform(
        lambda x: (x - x.iloc[0]) / x.iloc[0] * 100 if len(x) > 1 else 0
    )
    
    return growth_df

# Uso (added target_piastrina=1 for p1; change if needed)
if __name__ == "__main__":
    all_seeds, info, areas_df = process_all_piastrine(
        base_path=r"C:\Users\silve\Documents\Projects\Semi\piastrine",
        output_dir=r"C:\Users\silve\Documents\Projects\Semi\segmented_seeds_ordered",
        target_piastrina=1  
    )